
# 7. ÖZNİTELİK SEÇİMİ (FEATURE SELECTION)

## 7.1. Varyans ve Mutual Information Analizi

**Amaç:** Oluşturulan 64 giriş özniteliğinin saldırı sınıflandırması açısından bilgi değerini incelemek.

**Yöntem:**

İlk olarak eğitim kümesindeki sabit öznitelikler Varyans Eşiği (Variance Threshold) yaklaşımıyla belirlenecektir.

Ardından Mutual Information yöntemiyle özniteliklerin Attack_type hedef değişkeniyle ilişkisi hesaplanacaktır.

İşlem maliyetini azaltmak için eğitim kümesinden tabakalı rastgele örnekleme ile 80.000 kayıt seçilecektir.

**Beklenen Çıktı:**

- Sabit özniteliklerin sayısı ve listesi
- Mutual Information puanları
- En yüksek puanlı 20 özniteliğin sıralaması

**Not:** Analiz yalnızca eğitim verisi üzerinde gerçekleştirilecektir. Test verisi öznitelik seçiminde kullanılmayacaktır.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

# Eğitim veri setini okuma
klasor = Path(
    r"E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri"
)

dosya = klasor / "EdgeIIoT_Train_Muhendislik.csv"
df = pd.read_csv(dosya)

# Giriş ve hedef değişkenleri
X = df.drop(columns=["Attack_label", "Attack_type"])
y = df["Attack_type"]

print("Eğitim kaydı:", len(X))
print("Başlangıç özniteliği:", X.shape[1])

# 1. Sabit öznitelikleri belirleme
sabit = X.columns[X.nunique() <= 1].tolist()
X = X.drop(columns=sabit)

print("Sabit öznitelik:", len(sabit))
print("Sabit sütunlar:", sabit)
print("Kalan öznitelik:", X.shape[1])

# 2. Tabakalı örnekleme
ornek_idx, _ = train_test_split(
    np.arange(len(X)),
    train_size=80000,
    stratify=y,
    random_state=42
)

X_ornek = X.iloc[ornek_idx]
y_ornek = y.iloc[ornek_idx]

# 3. Ayrık öznitelikleri belirleme
on_ekler = (
    "http.request.method_",
    "http.request.version_",
    "mqtt.protoname_"
)

yeni_ikili = {
    "tcp_port_cifti_var",
    "tcp_hedef_bilinen_port",
    "http_istegi_var"
}

ayrik = [
    s.endswith("_eksik")
    or s.startswith(on_ekler)
    or s in yeni_ikili
    for s in X.columns
]

# 4. Mutual Information hesaplama
mi = mutual_info_classif(
    X_ornek,
    y_ornek,
    discrete_features=ayrik,
    random_state=42
)

mi_tablosu = pd.DataFrame({
    "Öznitelik": X.columns,
    "MI Puanı": mi
}).sort_values(
    "MI Puanı",
    ascending=False
).reset_index(drop=True)

print("\nEN YÜKSEK PUANLI 20 ÖZNİTELİK:")
display(mi_tablosu.head(20))

# Sonuçları kaydetme
mi_tablosu.to_csv(
    klasor / "07_Mutual_Information_Sonuclari.csv",
    index=False
)

assert len(df) == 319993
assert X.shape[1] + len(sabit) == 64

print("\nAnaliz tamamlandı.")


Eğitim kaydı: 319993
Başlangıç özniteliği: 64
Sabit öznitelik: 2
Sabit sütunlar: ['icmp.unused', 'mqtt.msg_decoded_as']
Kalan öznitelik: 62

EN YÜKSEK PUANLI 20 ÖZNİTELİK:


,Öznitelik,MI Puanı
0,tcp.dstport,0.954426
1,tcp.srcport,0.862856
2,tcp.ack,0.828708
3,mqtt.protoname_0.0,0.652871
4,tcp.seq,0.593349
5,tcp_port_cifti_var,0.567741
6,mqtt.protoname_0,0.551117
7,udp.stream,0.427919
8,tcp.flags.ack,0.332228
9,tcp.len,0.315227



Analiz tamamlandı.



### 7.1. Varyans ve Mutual Information Analizi — Sonuç ve Yorum

319.993 eğitim kaydı ve 64 giriş özniteliği üzerinde öznitelik seçimi analizi gerçekleştirilmiştir.

**Elde Edilen Bulgular:**

- Başlangıç özniteliği: 64
- Sabit öznitelik: 2
- Kalan öznitelik: 62
- Mutual Information için kullanılan örnek: 80.000 eğitim kaydı

Sabit olduğu belirlenen `icmp.unused` ve `mqtt.msg_decoded_as` öznitelikleri analizden çıkarılmıştır.

**En Yüksek Mutual Information Puanları:**

| Öznitelik | MI Puanı |
|---|---:|
| tcp.dstport | 0,9544 |
| tcp.srcport | 0,8629 |
| tcp.ack | 0,8287 |
| mqtt.protoname_0.0 | 0,6529 |
| tcp.seq | 0,5933 |
| tcp_port_cifti_var | 0,5677 |

**Değerlendirme:**

TCP portları ve bazı TCP protokol özniteliklerinin saldırı türleriyle yüksek bilgi ilişkisi gösterdiği belirlenmiştir.

Feature Engineering aşamasında oluşturulan `tcp_port_cifti_var` özniteliği de yüksek puanlı özellikler arasında yer almıştır.

Ancak yüksek Mutual Information puanı, özniteliğin model performansını mutlaka artıracağını garanti etmez. Veri toplama biçimlerinden kaynaklanabilecek yapay ilişkiler de değerlendirmede dikkate alınmalıdır.

**Sonuç:**

Sabit öznitelikler belirlenmiş ve kalan 62 öznitelik Mutual Information puanlarına göre sıralanmıştır.

Analiz yalnızca eğitim verileriyle gerçekleştirilmiştir. Sonraki aşamada yüksek puanlı özniteliklerden bir alt küme oluşturulacaktır.



## 7.2. Özniteliklerin Seçilmesi ve Veri Setlerinin Oluşturulması

**Amaç:** Mutual Information sonuçlarına göre yüksek bilgi değerine sahip özniteliklerden bir model veri kümesi oluşturmak.

**Yöntem:** Eğitim verilerinden hesaplanan Mutual Information sıralamasındaki ilk 20 öznitelik seçilecektir.

Aynı öznitelikler eğitim ve test kümelerinde korunacak, hedef değişkenler ayrıca saklanacaktır.

**Beklenen Çıktı:**

- 20 giriş özniteliği
- 2 hedef değişken
- Toplam 22 sütun içeren eğitim ve test CSV dosyaları

**Not:** İlk 20 öznitelik, model karşılaştırması için aday alt kümedir. Seçimin yararlılığı sonraki model değerlendirme aşamasında araştırılacaktır.


In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

# Çalışma klasörü
klasor = Path(
    r"E:\Data mining\Network_Intrusion_Detection\Calisma_Verileri"
)

# Önceki Mutual Information sonuçları
mi_dosyasi = klasor / "07_Mutual_Information_Sonuclari.csv"

mi_tablosu = pd.read_csv(mi_dosyasi)

# En yüksek puanlı 20 özniteliği seçme
secili_oznitelikler = (
    mi_tablosu.head(20)["Öznitelik"].tolist()
)

hedefler = ["Attack_label", "Attack_type"]
secili_sutunlar = secili_oznitelikler + hedefler

print("Seçilen öznitelik sayısı:", len(secili_oznitelikler))
print("\nSEÇİLEN 20 ÖZNİTELİK:")
display(mi_tablosu.head(20))

# Eğitim ve test veri kümelerini oluşturma
for grup, beklenen in [("Train", 319993), ("Test", 79999)]:

    giris = klasor / f"EdgeIIoT_{grup}_Muhendislik.csv"
    cikis = klasor / f"EdgeIIoT_{grup}_Secilmis.csv"

    ilk_parca = True
    toplam = 0

    for parca in pd.read_csv(
        giris,
        usecols=secili_sutunlar,
        dtype=str,
        na_filter=False,
        chunksize=20000
    ):

        # Eğitim ve test için aynı sütun sırası
        parca = parca[secili_sutunlar]

        # Seçilen giriş özniteliklerini doğrulama
        sayisal = parca[secili_oznitelikler].apply(
            pd.to_numeric,
            errors="coerce"
        )

        assert np.isfinite(
            sayisal.to_numpy(dtype=float)
        ).all(), "Sayısal olmayan değer bulundu!"

        parca.to_csv(
            cikis,
            mode="w" if ilk_parca else "a",
            header=ilk_parca,
            index=False
        )

        toplam += len(parca)
        ilk_parca = False

    assert toplam == beklenen

    print(f"\n{grup} kayıt sayısı:", toplam)
    print("Toplam sütun:", len(secili_sutunlar))
    print("Dosya:", cikis.name)

# Seçilen öznitelik listesini ayrıca kaydetme
pd.DataFrame({
    "Seçilen Öznitelik": secili_oznitelikler
}).to_csv(
    klasor / "07_Secilen_Oznitelikler.csv",
    index=False
)

assert len(secili_oznitelikler) == 20
assert len(secili_sutunlar) == 22

print("\nÖznitelik seçimi tamamlandı.")


Seçilen öznitelik sayısı: 20

SEÇİLEN 20 ÖZNİTELİK:


,Öznitelik,MI Puanı
0,tcp.dstport,0.954426
1,tcp.srcport,0.862856
2,tcp.ack,0.828708
3,mqtt.protoname_0.0,0.652871
4,tcp.seq,0.593349
5,tcp_port_cifti_var,0.567741
6,mqtt.protoname_0,0.551117
7,udp.stream,0.427919
8,tcp.flags.ack,0.332228
9,tcp.len,0.315227



Train kayıt sayısı: 319993
Toplam sütun: 22
Dosya: EdgeIIoT_Train_Secilmis.csv

Test kayıt sayısı: 79999
Toplam sütun: 22
Dosya: EdgeIIoT_Test_Secilmis.csv

Öznitelik seçimi tamamlandı.



### 7.2. Özniteliklerin Seçilmesi — Sonuç ve Yorum

Feature Selection aşamasında 64 giriş özniteliği değerlendirilmiştir.

**Elde Edilen Sonuçlar:**

- Başlangıç özniteliği: 64
- Sabit öznitelik: 2
- Mutual Information ile sıralanan öznitelik: 62
- Seçilen öznitelik: 20
- Öznitelik sayısındaki azalma: %68,75

**Oluşturulan Veri Kümeleri:**

| Veri Kümesi | Kayıt Sayısı | Sütun Sayısı |
|---|---:|---:|
| Eğitim | 319.993 | 22 |
| Test | 79.999 | 22 |

**Değerlendirme:**

Mutual Information puanlarına göre en yüksek bilgi değerine sahip 20 öznitelik seçilmiştir.

Öznitelik seçimi yalnızca eğitim verileri kullanılarak gerçekleştirilmiş, aynı sütunlar test veri setinde korunmuştur.

**Sonuç:**

Feature Selection aşaması tamamlanmıştır. Seçilen 20 öznitelik, makine öğrenmesi modellerinin eğitilmesinde kullanılacaktır.

Bu seçim henüz nihai optimum olarak değerlendirilmemektedir. Model sonuçları, öznitelik seçiminin etkisini incelemek amacıyla karşılaştırılacaktır.
